# Phase 6 PLAN 2 — DAEAC pseudo-label filtering
Four preregistered ablations. Checkpoints are selected only with source-validation Macro-F1; target evaluation runs after training.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
REPO_URL = 'https://github.com/CHANGE_ME/Best-thesis-in-council.git'
BRANCH = 'main'
WORK = Path('/kaggle/working')
ROOT = WORK / 'Best-thesis-in-council'
REPO = ROOT / 'ecg_thesis'
RUN_FULL = False
ENABLE_WANDB = False
RESUME_CHECKPOINTS = {}  # variant -> /kaggle/input/.../latest.pt
VARIANTS = ['no_filter', 'confidence_global', 'confidence_entropy', 'class_specific']
if not REPO.exists():
    assert 'CHANGE_ME' not in REPO_URL, 'Set REPO_URL first.'
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(ROOT)], check=True)
print(REPO)

## Install only repository requirements

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)

## Copy immutable Kaggle inputs into `/kaggle/working`

In [ ]:
INPUT = Path('/kaggle/input')
DATA_DEST = REPO / 'data/processed/phase6_daeac_paper'
CKPT_DEST = REPO / 'outputs/phase6_daeac_paper/checkpoints'
DATA_DEST.mkdir(parents=True, exist_ok=True); CKPT_DEST.mkdir(parents=True, exist_ok=True)
required = ['mitdb_ds1_daeac.npz', 'mitdb_ds2_first5_unlabeled_daeac.npz', 'mitdb_ds2_daeac.npz']
optional = ['incart_all_daeac.npz', 'svdb_all_daeac.npz']
def locate(name):
    matches = list(INPUT.rglob(name))
    if len(matches) != 1: raise FileNotFoundError(f'Expected exactly one {name}, found {matches}')
    return matches[0]
for name in required: shutil.copy2(locate(name), DATA_DEST / name)
for name in optional:
    matches = list(INPUT.rglob(name))
    if len(matches) == 1: shutil.copy2(matches[0], DATA_DEST / name)
base = locate('daeac_base_best.pt')
shutil.copy2(base, CKPT_DEST / 'daeac_base_best.pt')
print('Inputs ready:', DATA_DEST)

## Build disjoint after-five-minute target test, then run static/unit/protocol checks

In [ ]:
def run(args): subprocess.run([str(x) for x in args], cwd=REPO, check=True)
first_cfg = 'configs/phase6_daeac_pseudo_filter_no_filter.yaml'
run([sys.executable, 'scripts/phase6_daeac_pseudo_filter/00_prepare_after5.py', '--config', first_cfg])
run([sys.executable, 'scripts/check_repo.py'])
run([sys.executable, '-m', 'unittest', 'tests.test_daeac_pseudo_filter', 'tests.test_daeac_prototype_bank', 'tests.test_daeac_prototype_bank_training'])
for variant in VARIANTS:
    cfg = f'configs/phase6_daeac_pseudo_filter_{variant}.yaml'
    run([sys.executable, 'scripts/phase6_daeac_pseudo_filter/01_validate.py', '--config', cfg, '--strict'])

## One-epoch smoke tests (separate outputs; never used as full results)

In [ ]:
for variant in VARIANTS:
    cfg = f'configs/phase6_daeac_pseudo_filter_{variant}.yaml'
    out = f'/kaggle/working/smoke/pseudo_filter_{variant}'
    run([sys.executable, 'scripts/phase6_daeac_pseudo_filter/02_train.py', '--config', cfg, '--epochs', '1', '--max-source-samples', '512', '--max-target-samples', '512', '--max-val-samples', '512', '--output-dir', out, '--checkpoint-prefix', f'pseudo_filter_{variant}_smoke'])

## Full ablation (explicit opt-in; supports resume and optional W&B)

In [ ]:
if RUN_FULL:
    for variant in VARIANTS:
        cfg = f'configs/phase6_daeac_pseudo_filter_{variant}.yaml'
        cmd = [sys.executable, 'scripts/phase6_daeac_pseudo_filter/02_train.py', '--config', cfg]
        if variant in RESUME_CHECKPOINTS: cmd += ['--resume-checkpoint', RESUME_CHECKPOINTS[variant]]
        if ENABLE_WANDB: cmd += ['--wandb-enabled', '--wandb-project', 'ecg-thesis']
        run(cmd)
else:
    print('Set RUN_FULL=True after all smoke tests pass.')

## Post-training evaluation and comparison only

In [ ]:
if RUN_FULL:
    for variant in VARIANTS:
        cfg = f'configs/phase6_daeac_pseudo_filter_{variant}.yaml'
        ckpt = REPO / f'outputs/phase6_daeac_pseudo_filter_{variant}/checkpoints/daeac_pseudo_filter_{variant}_best.pt'
        run([sys.executable, 'scripts/phase6_daeac_pseudo_filter/03_eval.py', '--config', cfg, '--checkpoint', ckpt, '--dataset', 'all'])
    report_dir = WORK / 'phase6_daeac_pseudo_filter_report'
    run([sys.executable, 'scripts/phase6_daeac_pseudo_filter/04_make_report.py', '--config', first_cfg, '--output-dir', report_dir])
    bundle = WORK / 'phase6_daeac_pseudo_filter_bundle'
    if bundle.exists(): shutil.rmtree(bundle)
    bundle.mkdir()
    shutil.copytree(report_dir, bundle / 'report')
    for variant in VARIANTS: shutil.copytree(REPO / f'outputs/phase6_daeac_pseudo_filter_{variant}', bundle / variant)
    shutil.make_archive(str(bundle), 'zip', bundle)
    print('/kaggle/working/phase6_daeac_pseudo_filter_bundle.zip')